# pyro_api

pyroapi dispatches every call to a backend, and its default backend is Pyro, so these examples need Pyro installed too:

```bash
pip install pyro-api pyro-ppl
```

In [ ]:
# Example 1: Introduction to pyroapi and Backend Dispatch
import pyro
import pyroapi
from pyroapi import pyro_backend
from pyroapi.testing import MODELS

print("Introduction to pyroapi - Backend Dispatch System:")
print("=" * 55)

# pyroapi ships a few small models that a backend can be checked against
print("Built-in test models for backend validation:")
for model_name in MODELS.keys():
    print(f"  - {model_name}")
print(f"\nTotal models available: {len(MODELS)}")
assert set(MODELS) == {"logistic_regression", "neals_funnel", "eight_schools", "beta_binomial"}, set(MODELS)

beta_binomial_config = MODELS['beta_binomial']()
print(f"\n'beta_binomial' model function: {beta_binomial_config['model'].__name__}")
assert callable(beta_binomial_config["model"]) and "model_args" in beta_binomial_config

# Each pyroapi module is a dispatcher that forwards attribute lookups to the selected backend
modules = ['distributions', 'handlers', 'infer', 'ops', 'optim', 'pyro']
for module in modules:
    print(f"  pyroapi.{module}: {type(getattr(pyroapi, module)).__name__}")
assert all(hasattr(pyroapi, module) for module in modules)

# With the Pyro backend selected, pyroapi.pyro.sample is Pyro's own sample function
with pyro_backend("pyro"):
    assert pyroapi.pyro.sample is pyro.sample
    assert pyroapi.distributions.Normal is pyro.distributions.Normal
print("\npyroapi.pyro.sample dispatches to pyro.sample under the 'pyro' backend")

In [ ]:
# Example 2: Running the built-in models on the Pyro backend
from pyroapi import handlers, pyro_backend
from pyroapi.testing import MODELS

print("Running the built-in models on the Pyro backend:")
print("=" * 55)

# The observed sample sites each model is expected to record
expected_observed = {
    "logistic_regression": {"obs"},
    "neals_funnel": set(),
    "eight_schools": {"obs"},
    "beta_binomial": {"binomial"},
}

with pyro_backend("pyro"):
    for model_name, model_factory in MODELS.items():
        config = model_factory()
        model_fn = config['model']
        model_args = config.get('model_args', ())
        model_kwargs = config.get('model_kwargs', {})

        # Record every sample statement the model makes
        trace = handlers.trace(model_fn).get_trace(*model_args, **model_kwargs)
        sites = {name: site for name, site in trace.nodes.items() if site["type"] == "sample"}
        observed = {name for name, site in sites.items() if site["is_observed"]}
        print(f"\n'{model_name}': {len(sites)} sample sites, observed: {sorted(observed)}")
        assert sites, model_name
        assert observed == expected_observed[model_name], (model_name, observed)

print("\nAll built-in models ran and recorded their observed sites")

In [ ]:
# Example 3: Exploring pyroapi Generic Interface Modules
import pyroapi

print("Exploring pyroapi Generic Interface Modules:")
print("=" * 45)

# Attribute lookups on each dispatcher resolve against the current backend (Pyro by default)
expected = {
    "distributions": {"Normal", "Beta", "Bernoulli"},
    "handlers": {"trace", "condition", "seed"},
    "infer": {"SVI", "Trace_ELBO", "MCMC"},
    "optim": {"Adam", "ClippedAdam"},
    "pyro": {"sample", "param", "plate"},
}
for module_name, names in expected.items():
    attrs = [attr for attr in dir(getattr(pyroapi, module_name)) if not attr.startswith('_')]
    print(f"pyroapi.{module_name}: {len(attrs)} public names, e.g. {', '.join(sorted(names))}")
    assert names <= set(attrs), (module_name, names - set(attrs))

ops_attrs = [attr for attr in dir(pyroapi.ops) if not attr.startswith('_')]
print(f"pyroapi.ops: {len(ops_attrs)} public names")
assert "exp" in ops_attrs and "log" in ops_attrs

In [ ]:
# Example 4: Backend Registration and Dispatch System
import pyro
import pyroapi
from pyroapi import pyro_backend, register_backend
from pyroapi.dispatch import DEFAULT_RNG_SEED

print("Backend Registration and Dispatch System:")
print("=" * 45)

# Entering pyro_backend seeds the random number generator with DEFAULT_RNG_SEED,
# so the first draw inside each block is the same
print(f"Default RNG seed: {DEFAULT_RNG_SEED}")
assert isinstance(DEFAULT_RNG_SEED, int)
draws = []
for _ in range(2):
    with pyro_backend("pyro"):
        draws.append(float(pyroapi.pyro.sample("x", pyroapi.distributions.Normal(0.0, 1.0))))
print(f"First draw in two separate blocks: {draws}")
assert draws[0] == draws[1]

# register_backend adds a named alias; pyro_backend can then select it by name.
# Modules the alias does not name keep their current backend.
register_backend("pyro_alias", {"pyro": "pyro"})
with pyro_backend("pyro_alias"):
    assert pyroapi.pyro.sample is pyro.sample
    assert pyroapi.distributions.Normal is pyro.distributions.Normal
print("Registered alias 'pyro_alias' dispatches to Pyro")

# Backends can also be given directly as module names; a name the backend lacks raises NotImplementedError
with pyro_backend(pyro="math"):
    try:
        pyroapi.pyro.sample
    except NotImplementedError as e:
        print(f"Missing attribute on the 'math' backend: {e}")
    else:
        raise AssertionError("pyroapi.pyro.sample resolved on a backend that has no sample")

# Handlers dispatch too: condition fixes the value of a sample site
with pyro_backend("pyro"):
    with pyroapi.handlers.condition(data={"x": 3.0}):
        value = pyroapi.pyro.sample("x", pyroapi.distributions.Normal(0.0, 1.0))
print(f"Conditioned sample: {float(value)}")
assert float(value) == 3.0

In [ ]:
# Example 5: Built-in model signatures and the GenericModule dispatcher
import pyroapi
from pyroapi.testing import MODELS
from pyroapi.dispatch import GenericModule

print("Built-in model signatures:")
print("=" * 45)

# (number of positional arguments, keyword argument names) for each model
expected = {
    "logistic_regression": (1, ["y"]),
    "neals_funnel": (1, []),
    "eight_schools": (2, ["y"]),
    "beta_binomial": (3, ["data"]),
}
for model_name, model_factory in MODELS.items():
    model_config = model_factory()
    model_fn = model_config['model']
    model_args = model_config.get('model_args', ())
    model_kwargs = model_config.get('model_kwargs', {})

    print(f"\n{model_name}:")
    print(f"  Function name: {model_fn.__name__}")
    print(f"  Arguments: {len(model_args)} positional, keyword: {sorted(model_kwargs)}")
    assert (len(model_args), sorted(model_kwargs)) == expected[model_name], model_name

# Every pyroapi module is an instance of GenericModule
print(f"\nGenericModule: {GenericModule}")
for module in ['distributions', 'handlers', 'infer', 'ops', 'optim', 'pyro']:
    assert isinstance(getattr(pyroapi, module), GenericModule), module